# 🎤 Speech Enhancement — Step 2: Train the U-Net

This notebook:
1. Mounts Google Drive
2. Loads the spectrogram datasets created in notebook 01
3. Trains the U-Net speech-enhancement model
4. Saves the best weights back to Drive

### Prerequisites
- Run `01_Data_Preparation.ipynb` first, so spectrograms exist in Drive.
- Enable GPU in Colab: **Runtime → Change runtime type → GPU (T4)**

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
%pip install -q librosa soundfile scikit-learn

In [ ]:
# ── 3. Clone repo & add src/ to path ──────────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/YOUR_USERNAME/speech_enhancement_alt.git'  # <── update
REPO_DIR = '/content/speech_enhancement_alt'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready ✓')

In [ ]:
# ── 4. Configure training ──────────────────────────────────────────────────
import config as C

# Override any config values here if needed:
# C.EPOCHS     = 20
# C.BATCH_SIZE = 16
# C.TRAINING_FROM_SCRATCH = True

print(f'Epochs          : {C.EPOCHS}')
print(f'Batch size      : {C.BATCH_SIZE}')
print(f'From scratch    : {C.TRAINING_FROM_SCRATCH}')
print(f'Spectrogram dir : {C.SPEC_DIR}')
print(f'Weights dir     : {C.WEIGHTS_DIR}')

In [ ]:
# ── 5. Verify GPU is available ────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if not gpus:
    print('⚠️  No GPU detected. Training will be slow. Enable GPU in Runtime settings.')

In [ ]:
# ── 6. Preview the U-Net architecture ─────────────────────────────────────
from model import build_unet
model = build_unet()
model.summary()

In [ ]:
# ── 7. Train ──────────────────────────────────────────────────────────────
from train import train_model
model, history = train_model()

In [ ]:
# ── 8. Plot loss curve ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

loss     = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 4))
plt.plot(loss,     label='Train loss')
plt.plot(val_loss, label='Val loss')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Huber loss (log)')
plt.title('Training curve')
plt.legend()
plt.tight_layout()
plt.show()

import os
best_weights = os.path.join(C.WEIGHTS_DIR, 'model_best.h5')
if os.path.exists(best_weights):
    print(f'Best weights saved to: {best_weights}')
else:
    print('model_best.h5 not found (training may not have improved from epoch 1)')